# 14. PDF 예제를 DB 저장 형태로 바꾸기

[프로젝트 적용] 수업의 딕셔너리·SQL Mapper 개념을 개인 자료에 적용합니다.
[프로젝트 추가] 표·각주·그림 연결을 여섯 테이블의 행으로 나눕니다.
이 노트북은 읽기 전용 DB 조회와 메모리 변환만 합니다. INSERT와 Storage 업로드는 없습니다.
2026-10-01 터미널 미리보기: 부모 4개·청크 11개·이미지 3개·연결 3개. 벡터 11개는 아직 미확정입니다.

In [ ]:
# [프로젝트 추가] 노트북 실행 위치와 관계없이 개인 파이썬 파일을 찾습니다.
from pathlib import Path
import sys
current = Path.cwd().resolve()
project_folder = next((p for p in [current, *current.parents]
                       if (p / 'src/car_search_rag/zzong_santafe_lag/db_rows.py').is_file()), None)
if project_folder is None:
    raise FileNotFoundError('프로젝트 폴더에서 실행하세요.')
src_path = str(project_folder / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)


## 1. 개인 연결로 테이블을 읽기

ZZONG_DB_URL은 Git에서 제외되는 src/car_search_rag/zzong_santafe_lag/.env 또는 환경변수에 둡니다. 비밀번호를 셀에 직접 쓰지 않습니다.
read_only=True는 이번 연결의 트랜잭션에서 DB 변경을 거부하도록 지정합니다. 팀 DB 설정을 변경하지 않습니다.

In [ ]:
# [프로젝트 적용] Mapper의 SELECT로 개인 테이블만 조회합니다.
from car_search_rag.zzong_santafe_lag.database import ManualRepository, PersonalSqlSession
repository = ManualRepository(PersonalSqlSession(read_only=True))
connection_report = repository.connection_report()
print('읽기 전용 상태:', connection_report['connection']['read_only'])
for row in connection_report['tables']:
    print(row['table_name'], '저장된 행:', row['row_count'])


## 2. 예제 기록을 DB 행으로 변환

부모는 한 주제의 전체 글이고 청크는 검색할 짧은 글입니다.
44쪽 점도 표·45쪽 차대번호를 선정하고, 43쪽 추천 오일 표와 각주를 문맥으로 함께 포함합니다. 기존 필수 연결을 보존하며 새로운 필수 연결은 임의로 만들지 않습니다.
전체 PDF를 읽어 기존 청킹 규칙을 적용하지만 임베딩 모델은 실행하지 않습니다. 완료되면 결과는 메모리에만 남습니다.

In [ ]:
# [프로젝트 추가] 기존 prepare() 결과를 새 행으로 복사하며 원문을 직접 바꾸지 않습니다.
from car_search_rag.zzong_santafe_lag.manual_service import ManualService
from car_search_rag.zzong_santafe_lag.db_rows import build_sample_rows
service = ManualService().prepare()
preview = build_sample_rows(service)
summary = preview.summary()
print('변환 건수:', summary['table_counts'])
print('임베딩이 필요한 청크:', summary['pending_embeddings'])
print('지금 바로 저장 가능한가요?:', summary['ready_for_insert'])


In [ ]:
# 변환할 검색용 글 전체와 출처를 읽습니다. raw_text는 별도로 그대로 보관됩니다.
for row in preview.tables['parent_records']:
    print('\n주제:', row['title'], '| PDF:', row['source_pages'])
    print('보존 원문 글자 수:', len(row['raw_text']))
    print('필수 연결:', row['metadata'].get('required_context_record_ids', []))
    print(row['content'])


In [ ]:
# 이미지 파일은 그대로 두고, 쪽수·내부 키·로컬 경로를 연결합니다.
for row in preview.tables['images']:
    print(row['pdf_page_number'], row['pdf_image_key'], row['local_path'])
# 실제 청크 글과 벡터 미확정 상태를 한 개 확인합니다.
first_chunk = preview.tables['chunks'][0]
print('\n청크 글:', first_chunk['content'])
print('토큰 수:', first_chunk['token_count'])
print('벡터:', first_chunk['embedding'])


## 다음 단계

embedding=None은 아직 확정한 벡터가 없다는 뜻입니다. 0 벡터를 넣어 저장하지 않습니다.
다음에는 이 11개 청크를 기존 로컬 모델의 같은 버전으로 임베딩하고, 글 식별값·벡터 차원·정규화·참조를 확인해야 합니다.
실제 DB 저장은 별도 승인 후 진행합니다. 현재 실행 파일에는 저장 명령이 없습니다.
미리보기를 다시 실행하면 UUID(연결 번호)가 새로 발급되므로 실제 저장할 때는 확정한 묶음을 사용해야 합니다.